# Task 3.2 - Thuc nghiem dung / on dinh / hieu qua tren 2 bang trong de
Checker doc lap (`checker.py`) xac nhan moi loi giai. Dau vao doc tu `puzzles/*.json`.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # chay notebook tu thu muc experiments/
import statistics
from task3_kakuro.checker import KakuroChecker
from task3_kakuro.ablation_solver import KakuroCSPSolver
from task3_kakuro.puzzle_io import load_puzzle
PUZ = os.path.abspath('../task3_kakuro/puzzles')
CONFIGS = [('1. Backtracking thuan', {}), ('2. BT + Forward Checking', dict(use_fc=True)),
           ('3. BT + FC + Sum Propagation', dict(use_fc=True, use_prop=True)),
           ('4. BT + FC + Prop + MRV/Deg/LCV', dict(use_fc=True, use_prop=True, use_heuristics=True))]

In [3]:
def ablation(name):
    white, clues = load_puzzle(os.path.join(PUZ, name))
    ck = KakuroChecker(white, clues)
    print(name, '-', len(white), 'o trang,', len(clues), 'run')
    print(f"{'Cau hinh':<34} | {'Valid':<5} | {'Nodes':>7} | {'Backtracks':>10} | {'Time (ms)':>9}")
    print('-' * 78)
    for label, cfg in CONFIGS:
        r = KakuroCSPSolver(white, clues, **cfg).solve()
        ok = ck.check_solution(r['solution'])[0] if r['solution'] else False
        print(f"{label:<34} | {str(ok):<5} | {r['nodes']:7d} | {r['backtracks']:10d} | {r['time_ms']:9.3f}")
    print()
ablation('assignment_5x5.json')
ablation('assignment_9x9.json')

assignment_5x5.json - 12 o trang, 8 run
Cau hinh                           | Valid |   Nodes | Backtracks | Time (ms)
------------------------------------------------------------------------------
1. Backtracking thuan              | True  |   39093 |       4337 |    42.324
2. BT + Forward Checking           | True  |   26999 |       4337 |    37.177
3. BT + FC + Sum Propagation       | True  |      36 |         24 |     0.586
4. BT + FC + Prop + MRV/Deg/LCV    | True  |      12 |          0 |     0.639

assignment_9x9.json - 48 o trang, 28 run
Cau hinh                           | Valid |   Nodes | Backtracks | Time (ms)
------------------------------------------------------------------------------
1. Backtracking thuan              | True  |  126159 |      13990 |   220.632
2. BT + Forward Checking           | True  |   67153 |      13974 |   162.126
3. BT + FC + Sum Propagation       | True  |     234 |        186 |    10.283
4. BT + FC + Prop + MRV/Deg/LCV    | True  |     320 |    

## On dinh
Chay lai moi cau hinh 30 lan tren moi bang: bao trung binh va do lech chuan cua thoi gian; so node/backtrack la tat dinh (khong ngau nhien) nen phai giong nhau moi lan.

In [5]:
def stability(name, runs=30):
    white, clues = load_puzzle(os.path.join(PUZ, name))
    print(name)
    for label, cfg in CONFIGS:
        ts, nodes = [], set()
        for _ in range(runs):
            r = KakuroCSPSolver(white, clues, **cfg).solve()
            ts.append(r['time_ms']); nodes.add((r['nodes'], r['backtracks']))
        print(f"  {label:<34} mean={statistics.mean(ts):9.3f} ms  std={statistics.stdev(ts):8.3f} ms  node/backtrack khac nhau: {len(nodes)} gia tri")
stability('assignment_5x5.json')
stability('assignment_9x9.json')

assignment_5x5.json
  1. Backtracking thuan              mean=   41.661 ms  std=   3.178 ms  node/backtrack khac nhau: 1 gia tri
  2. BT + Forward Checking           mean=   38.227 ms  std=   5.219 ms  node/backtrack khac nhau: 1 gia tri
  3. BT + FC + Sum Propagation       mean=    0.562 ms  std=   0.036 ms  node/backtrack khac nhau: 1 gia tri
  4. BT + FC + Prop + MRV/Deg/LCV    mean=    0.336 ms  std=   0.022 ms  node/backtrack khac nhau: 1 gia tri
assignment_9x9.json
  1. Backtracking thuan              mean=  194.958 ms  std=   6.833 ms  node/backtrack khac nhau: 1 gia tri
  2. BT + Forward Checking           mean=  169.897 ms  std=  13.640 ms  node/backtrack khac nhau: 1 gia tri
  3. BT + FC + Sum Propagation       mean=   10.338 ms  std=   0.175 ms  node/backtrack khac nhau: 1 gia tri
  4. BT + FC + Prop + MRV/Deg/LCV    mean=   24.051 ms  std=   2.855 ms  node/backtrack khac nhau: 1 gia tri
